# Final holdout evaluation and freeze readiness

The formal **20,000-row holdout** is reserved for one corroborating evaluation after the complete development pipeline is frozen. Current evidence freezes no final model, probability source or prioritization policy, so **Run All completes readiness checks and leaves holdout evaluation disabled**. A numerical AP leader is not a freeze declaration.

The evaluator below is implemented for a later trusted frozen pipeline; it does not create or train that pipeline. Missing artifacts are reported as unmet prerequisites, not invented results. The historical preliminary audit exposed aggregate holdout-related information, so any eventual result must be described as corroboration, not a guaranteed pristine test.

## 1. Safe setup

Resolve the project root and read configuration metadata. The readiness path does not call the development loader, read raw employee data, deserialize a model or create the one-time holdout result directory.

In [1]:
from pathlib import Path
import json
import sys

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display

# Find the project from either its root or this notebook's directory.
ROOT = next((path for path in (Path.cwd().resolve(), *Path.cwd().resolve().parents)
             if (path / "configs/analysis_registry.yaml").is_file()), None)
if ROOT is None:
    raise FileNotFoundError("Run this notebook from inside the Employee Resignation Prediction project.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.evaluation.inputs import load_group_inputs, load_development_data
from src.evaluation.metrics import calculate_binary_metrics
from src.data.feature_contract import load_feature_contract

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
pd.set_option("display.float_format", lambda value: f"{value:.6f}")
contract = load_feature_contract(ROOT / "configs/feature_contract.yaml")

In [2]:
from src.evaluation.holdout import holdout_readiness, run_final_holdout

# Keep False during ordinary Run All. Setting True cannot bypass missing freeze checks.
RUN_FINAL_HOLDOUT = False
readiness, freeze = holdout_readiness(ROOT)
display(readiness)
ready = bool(readiness.passed.all())
print(f"All recorded freeze prerequisites satisfied: {ready}")
print("Readiness inspection has not read holdout features, targets or predictions.")

,check,passed,detail
0,comparison decision,True,results/comparison/comparison_summary.json
1,calibration decision,True,results/calibration/calibration_decision.json
2,fairness decision,True,results/fairness/fairness_summary.json
3,prioritization decision,True,results/prioritization/prioritization_decision...
4,final model frozen,False,A numerical OOF leader alone is not a final fr...
5,no previous assessment,True,Any receipt or result blocks reruns; an empty ...
6,valid freeze manifest,False,Missing or invalid: models/frozen/freeze.json


All recorded freeze prerequisites satisfied: False
Readiness inspection has not read holdout features, targets or predictions.


## 2. Required development freeze

Before intentional evaluation, the group must select and record a final model using development evidence, finalize preprocessing and calibration (including an explicit raw-probability decision if justified), and freeze the threshold and downstream policy. Save a complete development-fitted pipeline or project adapter with `joblib` in `models/frozen/`. It must expose `predict_proba` on the exact approved feature DataFrame and already include every chosen transformation. A sklearn classifier exposes `classes_`; project adapters may return a 1D positive-class vector.

Create `models/frozen/freeze.json` **before holdout access** with the schema displayed below. Training partition and row count are recorded group attestations; hashes bind artifacts but cannot reconstruct training provenance from weights. The comparison decision must name `final_model` and set `final_model_frozen=true`. Calibration must set `probability_source_frozen=true` and declare `probability_source` as raw, sigmoid or isotonic. Prioritization must set `policy_frozen=true`, with an explicit decision to retain or omit the optional simulation. This notebook freezes none of these decisions itself.

In [3]:
# Schema illustration only: no manifest or fitted artifact is written from this example.
freeze_schema = {
    "freeze_id": "group-approved-development-freeze-id",
    "pipeline_frozen": True, "model": "explicit final_model from comparison decision",
    "training_partition": "development", "training_rows": contract["split_registry"]["development_rows"],
    "feature_columns": contract["primary_model_features"],
    "pipeline_path": "models/frozen/pipeline.joblib",
    "probability_source": "raw OR sigmoid OR isotonic",
    "includes_frozen_probability_source": True,
    "threshold": "a numeric cutoff fixed from development evidence",
    "threshold_source": "development-only selection method and rationale",
    "downstream_policy_frozen": True, "holdout_scope": "prediction_only", "formal_holdout_used_for_selection": False,
    "historical_audit_exposure_disclosed": True,
    "artifacts": {path: "exact-byte SHA-256 of the approved local file" for path in [
        "models/frozen/pipeline.joblib", "configs/feature_contract.yaml", "configs/analysis_registry.yaml",
        "results/comparison/comparison_summary.json", "results/calibration/calibration_decision.json",
        "results/fairness/fairness_summary.json", "results/prioritization/prioritization_decision.json",
    ]},
}
display(Markdown("```json\n" + json.dumps(freeze_schema, indent=2) + "\n```"))

```json
{
  "freeze_id": "group-approved-development-freeze-id",
  "pipeline_frozen": true,
  "model": "explicit final_model from comparison decision",
  "training_partition": "development",
  "training_rows": 80000,
  "feature_columns": [
    "Department",
    "Gender",
    "Job_Title",
    "Education_Level",
    "Age",
    "Years_At_Company",
    "Performance_Score",
    "Monthly_Salary",
    "Work_Hours_Per_Week",
    "Projects_Handled",
    "Overtime_Hours",
    "Sick_Days",
    "Remote_Work_Frequency",
    "Team_Size",
    "Training_Hours",
    "Promotions",
    "Employee_Satisfaction_Score"
  ],
  "pipeline_path": "models/frozen/pipeline.joblib",
  "probability_source": "raw OR sigmoid OR isotonic",
  "includes_frozen_probability_source": true,
  "threshold": "a numeric cutoff fixed from development evidence",
  "threshold_source": "development-only selection method and rationale",
  "downstream_policy_frozen": true,
  "holdout_scope": "prediction_only",
  "formal_holdout_used_for_selection": false,
  "historical_audit_exposure_disclosed": true,
  "artifacts": {
    "models/frozen/pipeline.joblib": "exact-byte SHA-256 of the approved local file",
    "configs/feature_contract.yaml": "exact-byte SHA-256 of the approved local file",
    "configs/analysis_registry.yaml": "exact-byte SHA-256 of the approved local file",
    "results/comparison/comparison_summary.json": "exact-byte SHA-256 of the approved local file",
    "results/calibration/calibration_decision.json": "exact-byte SHA-256 of the approved local file",
    "results/fairness/fairness_summary.json": "exact-byte SHA-256 of the approved local file",
    "results/prioritization/prioritization_decision.json": "exact-byte SHA-256 of the approved local file"
  }
}
```

## 3. One-time execution gate

Activation requires both `RUN_FINAL_HOLDOUT=True` and every readiness check to pass. The evaluator verifies frozen source/split checksums, loads only the trusted local pipeline, and exclusively creates an assessment receipt under `results/final_holdout/` before reading holdout labels or predicting. A started receipt remains after failures; an existing receipt or result blocks automatic retries or a second process. Empty reserved directories and `.gitkeep` are allowed. Do not delete assessment evidence to rerun after inspecting results.

Raw input is read by ID first, then only holdout rows are parsed for predictors and labels. Require all assigned holdout IDs exactly once. No fit, tuning, calibration selection, threshold search, proxy fitting or alternate-model comparison is allowed. Serialized joblib files must originate from the trusted local development freeze.

In [4]:
holdout_metrics = holdout_reliability = holdout_intervals = None
if RUN_FINAL_HOLDOUT:
    if not ready:
        raise RuntimeError("Holdout blocked: resolve the failed readiness checks before activation.")
    holdout_metrics, holdout_reliability, holdout_intervals = run_final_holdout(ROOT, authorized=True)
    display(pd.DataFrame([holdout_metrics]))
    display(holdout_intervals)
else:
    print("LOCKED: final holdout evaluation skipped. Readiness checks completed successfully.")

LOCKED: final holdout evaluation skipped. Readiness checks completed successfully.


## 4. Metrics and uncertainty after an approved evaluation

AP is the primary ranking metric; ROC-AUC describes discrimination. Precision, recall, F1, balanced accuracy and the confusion matrix use the **frozen threshold**, not a newly selected base-rate or maximum-F1 cutoff. Brier score and ten-bin ECE assess probability accuracy. Compare AP with holdout prevalence, ROC-AUC with 0.50, and probability error with a constant score fixed from development prevalence.

Report 95% percentile intervals from **2,000 class-stratified bootstrap resamples, seed 42**. They are conditional on this fitted model and fixed prediction set: they do not include retraining uncertainty, model selection uncertainty or dataset shift. Undefined subgroup rates must remain missing. Historical audit exposure remains a limitation.

In [5]:
if holdout_metrics is not None:
    display(holdout_reliability)
    comparison = pd.read_csv(ROOT / "results/comparison/model_comparison.csv")
    selected = comparison.loc[comparison.model == freeze["model"]]
    if len(selected) != 1:
        raise ValueError("The frozen model must have exactly one development comparison row.")
    development = selected.iloc[0]
    generalization = pd.DataFrame([
        {"metric": "Average Precision", "development_oof": development.pooled_average_precision,
         "holdout": holdout_metrics["average_precision"]},
        {"metric": "ROC-AUC", "development_oof": development.pooled_roc_auc,
         "holdout": holdout_metrics["roc_auc"]},
    ])
    generalization["holdout_minus_development"] = generalization.holdout - generalization.development_oof
    display(generalization)
    print("OOF and final full-development fits have different training sizes; gaps are descriptive.")
    print("If the frozen probability source is calibrated, raw OOF Brier is not a like-for-like calibration comparison.")
else:
    print("No holdout metrics or confidence intervals exist for this run; none are inferred from development results.")

No holdout metrics or confidence intervals exist for this run; none are inferred from development results.


## 5. Corroboration figures after evaluation

Show PR and ROC curves plus a reliability diagram only when an approved evaluation produced them. Curves are saved aggregate evidence and cannot be used to revise the chosen pipeline. The exported confusion counts preserve actual and predicted positive/negative denominators.

In [6]:
if holdout_metrics is not None:
    out = ROOT / "results/final_holdout"
    pr = pd.read_csv(out / "holdout_pr_curve.csv")
    roc = pd.read_csv(out / "holdout_roc_curve.csv")
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))
    axes[0].plot(pr.recall, pr.precision)
    axes[0].axhline(holdout_metrics["prevalence"], linestyle="--", color="gray", label="Holdout prevalence")
    axes[0].set(xlabel="Recall", ylabel="Precision", title="Frozen holdout PR curve", xlim=(0, 1), ylim=(0, 1))
    axes[0].legend(fontsize=8)
    axes[1].plot(roc.false_positive_rate, roc.true_positive_rate)
    axes[1].plot([0, 1], [0, 1], "k--")
    axes[1].set(xlabel="False-positive rate", ylabel="True-positive rate", title="Frozen holdout ROC", xlim=(0, 1), ylim=(0, 1))
    occupied = holdout_reliability.loc[holdout_reliability["count"] > 0]
    axes[2].plot(occupied.mean_probability, occupied.observed_rate, marker="o")
    axes[2].plot([0, 1], [0, 1], "k--")
    axes[2].set(xlabel="Mean predicted probability", ylabel="Observed rate", title="Frozen holdout reliability", xlim=(0, 1), ylim=(0, 1))
    fig.tight_layout()
    fig.savefig(out / "figure_01_holdout_corroboration.png", dpi=160)
    plt.show()
    plt.close(fig)
    confusion = pd.DataFrame([[holdout_metrics["tn"], holdout_metrics["fp"]],
                              [holdout_metrics["fn"], holdout_metrics["tp"]]],
                             index=["Actual 0", "Actual 1"], columns=["Predicted 0", "Predicted 1"])
    display(confusion)
else:
    print("Holdout figures skipped because evaluation is disabled.")

Holdout figures skipped because evaluation is disabled.


## 6. Downstream scope and honest completion status

This evaluator corroborates the frozen prediction pipeline. Development fairness and prioritization results do not become holdout results automatically. If the group retains downstream holdout corroboration, its subgroup, proxy normalization, capacities and strategy evaluator must be separately frozen before any holdout access. Do not fit MinMaxScaler to holdout data or search proxy strategies on holdout labels. This scope must be settled at the development freeze; no downstream holdout result is fabricated here.

While locked, export a **readiness report** under `results/holdout_readiness/`, separate from the exclusive final assessment directory. Completion means this notebook runs from a clean kernel, verifies the gate and accurately records unmet prerequisites. It does not mean the formal holdout was evaluated.

In [7]:
def save_verified_table(frame, path):
    """Export an aggregate table and verify its values survive the CSV round trip."""
    frame.to_csv(path, index=False)
    saved = pd.read_csv(path)
    pd.testing.assert_frame_equal(saved, frame.reset_index(drop=True), check_dtype=False,
                                  rtol=1e-10, atol=1e-12)


def save_verified_json(document, path):
    """Reject non-finite JSON values and confirm a readable, exact round trip."""
    path.write_text(json.dumps(document, indent=2, allow_nan=False), encoding="utf-8")
    assert json.loads(path.read_text(encoding="utf-8")) == document

In [8]:
readiness_out = ROOT / "results/holdout_readiness"
readiness_out.mkdir(parents=True, exist_ok=True)
status = {
    "readiness_checks_complete": True, "freeze_ready": ready,
    "evaluation_requested": RUN_FINAL_HOLDOUT,
    "formal_holdout_evaluated_this_run": holdout_metrics is not None,
    "status": "evaluated" if holdout_metrics is not None else "locked",
    "unmet_checks": readiness.loc[~readiness.passed, "check"].tolist(),
    "historical_audit_exposure_caveat": "Earlier preliminary audit exposed aggregate holdout-related information; eventual assessment is corroborative.",
}
save_verified_table(readiness, readiness_out / "freeze_readiness.csv")
save_verified_json(status, readiness_out / "holdout_status.json")
if holdout_metrics is None:
    assert not RUN_FINAL_HOLDOUT
    print("Complete: readiness report verified; formal holdout remains locked pending an actual development freeze.")
else:
    receipt = json.loads((ROOT / "results/final_holdout/assessment_receipt.json").read_text(encoding="utf-8"))
    assert receipt["status"] == "completed"
    print("Complete: one frozen-pipeline holdout assessment saved. No subsequent model or policy changes may use it.")
display(pd.DataFrame([status]))

Complete: readiness report verified; formal holdout remains locked pending an actual development freeze.


,readiness_checks_complete,freeze_ready,evaluation_requested,formal_holdout_evaluated_this_run,status,unmet_checks,historical_audit_exposure_caveat
0,True,False,False,False,locked,"[final model frozen, valid freeze manifest]",Earlier preliminary audit exposed aggregate ho...
